# FarmTech Solutions · Visão computacional para um rover autônomo de pasto

**FIAP · Inteligência Artificial · Fase 6 · Redes Neurais**
**Autor:** Jefferson Gonçalves Lemos · RM572399

> Módulo de percepção visual de um rover que inspeciona o pasto. Ele responde a três perguntas: onde está o rebanho, se há risco biológico no caminho e por onde o rover pode andar.

| Material | Link |
|---|---|
| Repositório (README) | ⏳ *link a inserir* |
| Vídeo de demonstração (YouTube, não listado) | ⏳ *link a inserir* |
| Notebook do Ir Além (Raspberry Pi 5 + Hailo-8L) | ⏳ *link a inserir* |

## Sumário

1. [Contexto e decisões de projeto](#1)
2. [Setup do ambiente](#2)
3. [Dataset](#3)
4. [Entrega 1 · Treino da YOLOv5 customizada (30 × 60 épocas)](#4)
5. [Entrega 1 · Validação e teste](#5)
6. [Entrega 1 · Inferência nas imagens de teste](#6)
7. [Entrega 1 · Conclusões](#7)
8. [Entrega 2a · YOLO tradicional (COCO, sem customização)](#8)
9. [Entrega 2b · CNN treinada do zero](#9)
10. [Entrega 2 · Extra: YOLOv8s no mesmo dataset](#10)
11. [Protocolo comum de tempo de inferência](#11)
12. [Entrega 2 · Comparação final](#12)
13. [Conclusões gerais, pontos fortes e limitações](#13)

**Como executar:** selecione *Ambiente de execução → Alterar o tipo de ambiente → GPU* e depois *Executar tudo*. Qualquer GPU do Colab atende (T4 ou superior); a placa alocada fica registrada na saída da primeira célula de código. Publiquei o dataset em um link público, de modo que quem for executar o notebook não precisa de acesso ao meu Google Drive. O Drive só é usado, opcionalmente, para guardar os resultados (flag `SALVAR_NO_DRIVE`). A flag `ENSAIO` (seção 2) reduz as épocas para validar o pipeline em poucos minutos; ela fica desligada na execução final, e os números obtidos em modo ensaio não são os resultados do trabalho. A flag `RODAR_YOLOV8` controla a comparação extra com a YOLOv8s (seção 10); desligada, o notebook compara apenas os três modelos do enunciado. Também é possível enviar `farmtech_fase6_dataset.zip` para `/content` pelo painel de arquivos do Colab; nesse caso o notebook usa o zip local e não baixa pelo link.

<a id="1"></a>
## 1. Contexto e decisões de projeto

### O cliente e o problema

A FarmTech está prototipando um **rover autônomo de inspeção de pasto**. O módulo de visão que desenvolvi responde a três perguntas:

| Pergunta | Classe | Ação do rover |
|---|---|---|
| Onde está o rebanho? | `vaca` | contar e registrar a posição |
| Há risco biológico no caminho? | `cobra`, `escorpiao` | parar e alertar |
| Por onde o rover pode andar? | `arvore`, `cerca` | desviar e respeitar o limite do pasto |
| (demonstração ao vivo do Ir Além) | `caneca` | nenhuma no pasto; serve para testar o fluxo câmera → Wi-Fi → modelo |

O enunciado pede no mínimo dois objetos bem diferentes (A e B), com 40 imagens cada e divisão 32/4/4. Trabalhei com **seis classes**, e cada uma tem pelo menos 32 imagens de treino, 4 de validação e 4 de teste. Como uma foto de pasto costuma mostrar vaca, árvore e cerca ao mesmo tempo, a mesma imagem conta para mais de uma classe.

### Classe `caneca`, para a demonstração ao vivo

A sexta classe, `caneca`, não responde a nenhuma pergunta do rover. Incluí essa classe para viabilizar a demonstração ao vivo do Ir Além, em que o celular funciona como câmera Wi-Fi e o `best.pt` roda no VS Code. No ambiente onde gravei a demonstração não há vacas, cobras nem escorpiões, e uma caneca é um objeto comum, fácil de colocar diante da câmera e de mover pela cena. Assim a demonstração mostra o modelo customizado detectando um objeto real, ao vivo, e não apenas imagens do dataset. Coloquei `caneca` no índice 5, depois das cinco classes do pasto, para não renumerar os rótulos que já existiam. As fotos vêm do Wikimedia Commons, podendo ser complementadas por fotos próprias, e passaram pela mesma rotulação e revisão das demais classes.

### Cerca rotulada no mourão

A cerca é uma estrutura linear, que costuma aparecer na diagonal da foto. Uma caixa em volta dela seria preenchida por cerca de 90% de capim e céu, e o modelo acabaria aprendendo o fundo. Por isso desenhei a caixa da classe `cerca` em cada **mourão** (o poste), que é compacto, vertical e tem proporção constante. Para o rover, uma fileira de mourões detectados já marca o limite do pasto.

### Rotulação: Grounding DINO + revisão manual no Roboflow (em vez do Make Sense)

O enunciado indica o Make Sense IA. Optei por outro fluxo, com o mesmo resultado final:

1. **Pré-rotulação** com o Grounding DINO, um detector *zero-shot* guiado por texto (`"a cow."`, `"a snake."`, `"a wooden fence post."`…);
2. **Revisão manual de todas as caixas** no Roboflow Annotate: ajustei as caixas, removi falsos positivos e incluí os objetos que o detector deixou passar;
3. Exportação no **mesmo formato YOLO** (um `.txt` por imagem) que o Make Sense gera.

Escolhi esse fluxo por motivos técnicos. São algumas centenas de instâncias em seis classes, e a pré-rotulação *zero-shot* reduz bastante o tempo de desenhar caixa por caixa. Como toda caixa passou pela minha revisão, a sugestão automática não entra no dataset sem conferência. O processo também pode ser repetido, porque os scripts estão em `rotulacao/` no repositório. Como a exportação segue o mesmo formato YOLO, as etapas de treino, validação e teste não mudam em relação ao que o capítulo descreve. Rotulei e revisei também a validação e o teste, porque sem rótulos no teste não existe métrica objetiva.

### Divisão sem vazamento

Se uma foto fosse para o treino e outra quase idêntica a ela fosse para o teste, a métrica ficaria inflada. Por isso o `split.py` distribui **grupos** de imagens, e não imagens soltas: o grupo inteiro vai para o mesmo conjunto, e o teste e a validação são preenchidos primeiro até o mínimo de 4 imagens por classe. O grupo é definido por uma expressão regular aplicada ao nome do arquivo (`--grupo`); quando ela não casa, a imagem forma um grupo próprio.

Esse agrupamento foi pensado para as fotos em sequência que o `rotulacao/coleta.py` captura com a câmera do Pi ou com uma webcam, cujo nome traz a data e a hora da caminhada. O dataset atual, porém, não usa essas fotos: o lote A vem do Wikimedia Commons (`pasto_commons_NNN.jpg`) e os lotes B e C vêm do iNaturalist, sem data e hora no nome. Por isso cada foto forma um grupo independente. Isso é adequado porque, de modo geral, fotos diferentes vêm de autores e locais diferentes. O risco que sobra é um mesmo autor ter publicado várias fotos da mesma cena e elas caírem em conjuntos diferentes; registro essa limitação na seção 13. A seção 3 confere que nenhum arquivo aparece em mais de um conjunto.

### Origem das imagens

| Lote | Classes | Origem |
|---|---|---|
| A | `vaca`, `arvore`, `cerca` | Wikimedia Commons (categorias *Cattle in Brazil*, *Pastures in Brazil*, *Wooden fences in Brazil*, *Fence posts* e *Barbed wire fences*, com as subcategorias diretas; JPEG com licença CC0 / CC-BY / CC-BY-SA ou domínio público e resolução mínima; créditos em `rotulacao/creditos.csv`) |
| B | `cobra` | iNaturalist (observações *research grade*, licenças CC0 / CC-BY / CC-BY-SA, com prioridade para o Brasil; créditos em `rotulacao/creditos.csv`) |
| C | `escorpiao` | iNaturalist (observações *research grade*, licenças CC0 / CC-BY / CC-BY-SA, com prioridade para o Brasil; créditos em `rotulacao/creditos.csv`) |
| D | `caneca` | Wikimedia Commons (categorias *Coffee mugs*, *Mugs with logos*, *People with mugs*, *Mugs with soup* e *Mugs*, com as subcategorias diretas; 60 fotos com cota por categoria, sem títulos de acervos de museu nem fotos de produto em fundo branco; licença CC0 / CC-BY / CC-BY-SA ou domínio público; créditos em `rotulacao/creditos.csv`), podendo ser complementado por fotos próprias |

Baixei as fotos do lote A com o `rotulacao/baixar_commons.py`, que percorre as cinco categorias de pasto do Commons listadas na tabela e filtra os arquivos por licença livre, formato JPEG e resolução mínima. Dividi o total em cotas por categoria (a sobra de uma categoria com poucas fotos válidas passa para as seguintes), porque só a categoria de gado já forneceria quase todo o lote, e eu precisava também de cercas, mourões e árvores em quantidade suficiente. São fotos de autores diferentes, feitas com câmeras, distâncias e enquadramentos variados, desde planos abertos do pasto até fotos mais próximas de cercas e mourões.

As 60 fotos do lote D vieram do mesmo script, apontado para as cinco categorias de caneca da tabela, com o mesmo filtro de licença livre e a mesma divisão em cotas por categoria. Nesse lote excluí os títulos de acervos de museu e as fotos de produto em fundo branco, porque mostram peças de coleção ou de catálogo, e não a caneca em uso que a câmera vê na demonstração ao vivo. Como nos lotes B e C, baixei apenas as imagens dos lotes A e D: toda a rotulação foi feita neste projeto.

Se alguma classe ficar abaixo do mínimo de 32/4/4 depois da revisão, posso complementá-la com imagens do Roboflow Universe, também baixando somente as imagens, sem as anotações.

<a id="2"></a>
## 2. Setup do ambiente

Usei o **YOLOv5** da Ultralytics, o mesmo do capítulo, que roda sobre PyTorch. A CNN da Entrega 2 usa TensorFlow/Keras, que já vem instalado no Colab. Para a comparação extra da seção 10 uso também a YOLOv8s do pacote `ultralytics`. O próprio `requirements.txt` do YOLOv5 já depende desse pacote; deixei-o explícito no `pip` apenas para registrar a dependência. Como o `pip` não usa `-U`, o PyTorch que o Colab já traz não é reinstalado.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv  # confirma qual GPU o Colab alocou

In [ ]:
# Clona o YOLOv5 (pula se já existir) e instala as dependências, o gdown (download do dataset público)
# e o ultralytics (YOLOv8 da comparação extra; sem -U, o PyTorch do Colab é mantido)
!git clone -q https://github.com/ultralytics/yolov5 /content/yolov5 2>/dev/null || echo "yolov5 já clonado"
!pip install -q -r /content/yolov5/requirements.txt gdown ultralytics

In [ ]:
import json, random, re, shutil, subprocess, sys, time, zipfile        # utilitários da biblioteca padrão
from collections import Counter                                        # contagens por classe
from pathlib import Path                                               # caminhos portáveis

import matplotlib.pyplot as plt                                        # gráficos
import numpy as np                                                     # vetores e métricas
import pandas as pd                                                    # tabelas de resultados
import torch                                                           # backend do YOLOv5
import yaml                                                            # escrita do data.yaml
from IPython.display import display                                    # mostra DataFrames no meio da célula
from PIL import Image, ImageDraw                                       # leitura de imagens e desenho de caixas

SEED = 42                                                              # semente única para todo o notebook
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)       # reprodutibilidade

CLASSES = ["vaca", "cobra", "escorpiao", "arvore", "cerca", "caneca"]  # ordem = índice nos .txt (cerca = mourão)
CORES = ["#1f77b4", "#d62728", "#9467bd", "#2ca02c", "#ff7f0e", "#c850c8"]  # uma cor por classe nos desenhos
MINIMO = {"train": 32, "val": 4, "test": 4}                             # imagens mínimas por classe (enunciado)
SPLITS = list(MINIMO)                                                  # ["train", "val", "test"]
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}                        # extensões de imagem aceitas

YOLO_DIR = Path("/content/yolov5")                                      # código do YOLOv5
DATA_DIR = Path("/content/dataset")                                     # dataset descompactado
RUNS = Path("/content/runs")                                            # saídas de treino, validação e detecção
ASSETS = Path("/content/assets")                                        # figuras para o README e o vídeo
for pasta in (RUNS, ASSETS):                                           # cria as pastas de saída
    pasta.mkdir(parents=True, exist_ok=True)                           # sem erro se já existirem

DATASET_ID = "<!-- TODO: id do Drive -->"                               # id público de farmtech_fase6_dataset.zip
SALVAR_NO_DRIVE = False                                                # True = copia runs/assets para o Drive do autor
DRIVE_DIR = Path("/content/drive/MyDrive/FarmTech_Fase6")              # destino no Drive (só com a flag acima)
RODAR_FREEZE = False                                                   # True = 3º experimento com backbone congelado
RODAR_YOLOV8 = True                                                    # True = treina a YOLOv8s da comparação extra (seção 10)
ENSAIO = False                                                         # True = ensaio rápido do pipeline para validar o notebook; não gera os resultados do trabalho
EPOCAS = (3, 6) if ENSAIO else (30, 60)                                # épocas das simulações 1 e 2 (reduzidas no modo ensaio)
EPOCAS_CNN = 10 if ENSAIO else 100                                     # limite de épocas da CNN (reduzido no modo ensaio)
IMG, BATCH = 640, 16                                                   # resolução de entrada e tamanho do lote
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"                # usa a GPU se houver
print(f"PyTorch {torch.__version__} · dispositivo: {DEVICE}")          # confirma o ambiente

In [ ]:
def rodar(cmd, cwd=YOLO_DIR):
    """Executa um script do YOLOv5, mostra a saída ao vivo e devolve (texto completo, segundos)."""
    t0 = time.perf_counter()                                           # inicia o cronômetro
    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, text=True,  # o YOLOv5 escreve o log em stderr,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)  # por isso os dois fluxos são unidos
    linhas = []                                                        # guarda a saída para extrair métricas depois
    for linha in proc.stdout:                                          # lê linha a linha enquanto o script roda
        linhas.append(linha)                                           # acumula o texto bruto
        print(linha.rsplit("\r", 1)[-1], end="")                       # a barra do tqdm usa \r: mostra só o estado final
    proc.wait()                                                        # espera o processo terminar
    if proc.returncode:                                                # código ≠ 0 = falha
        raise RuntimeError("falhou: " + " ".join(map(str, cmd)))       # interrompe o notebook com a causa
    return "".join(linhas), time.perf_counter() - t0                   # saída e duração em segundos


def salvar_drive(*pastas):
    """Copia as pastas para o Drive do autor, apenas se SALVAR_NO_DRIVE=True."""
    if not SALVAR_NO_DRIVE:                                            # sem a flag, o Drive não é usado
        return                                                         # nada a fazer
    from google.colab import drive                                     # só existe no Colab
    if not Path("/content/drive/MyDrive").exists():                    # monta uma única vez
        drive.mount("/content/drive")                                  # pede autorização da conta Google
    for pasta in pastas:                                               # copia cada pasta pedida
        shutil.copytree(pasta, DRIVE_DIR / pasta.name, dirs_exist_ok=True)  # sobrescreve versões antigas

<a id="3"></a>
## 3. Dataset

O dataset é baixado de um link público, para que o notebook rode sem acesso ao meu Google Drive. O zip tem `train/`, `val/` e `test/`, cada um com `images/` e `labels/` (formato YOLO: `classe xc yc w h`, normalizados entre 0 e 1). O `data.yaml` é regravado nesta seção, então o caminho sempre aponta para `/content/dataset`.

In [ ]:
ZIP = Path("/content/farmtech_fase6_dataset.zip")                      # destino do download
if not (DATA_DIR / "train").exists():                                  # baixa só na primeira execução
    if not ZIP.exists():                                               # pula o download se o zip já foi enviado para /content
        assert "TODO" not in DATASET_ID, "Defina DATASET_ID com o id público do zip no Drive"  # falha clara
        !gdown $DATASET_ID -O $ZIP  # baixa o zip público (ID posicional; --id foi descontinuado)
    with zipfile.ZipFile(ZIP) as z:                                    # abre o zip
        z.extractall(DATA_DIR)                                         # extrai em /content/dataset
    raiz = next(p.parent for p in DATA_DIR.rglob("train") if (p / "images").is_dir())  # acha onde está train/
    if raiz != DATA_DIR:                                               # o zip veio com uma pasta a mais
        for item in raiz.iterdir():                                    # sobe o conteúdo um nível
            shutil.move(str(item), DATA_DIR / item.name)               # train/, val/, test/ na raiz

DATA_YAML = DATA_DIR / "data.yaml"                                     # configuração lida pelo YOLOv5
DATA_YAML.write_text(yaml.safe_dump({"path": str(DATA_DIR), "train": "train/images", "val": "val/images",
                                     "test": "test/images", "nc": len(CLASSES), "names": CLASSES},
                                    sort_keys=False, allow_unicode=True))  # sempre coerente com CLASSES
print(DATA_YAML.read_text())                                           # mostra a configuração final

In [ ]:
def imagens(split):
    """Lista ordenada das imagens de um split."""
    return sorted(p for p in (DATA_DIR / split / "images").iterdir() if p.suffix.lower() in EXT)


def rotulo_de(img):
    """Caminho do .txt correspondente a uma imagem."""
    return img.parent.parent / "labels" / f"{img.stem}.txt"


def ler_rotulos(txt):
    """Lê um .txt YOLO e devolve um array (n, 5): classe, xc, yc, w, h."""
    if not txt.exists():                                               # imagem sem rótulo
        return np.zeros((0, 5))                                        # nenhuma caixa
    linhas = [l.split() for l in txt.read_text().splitlines() if l.strip()]  # ignora linhas vazias
    return np.array(linhas, dtype=float).reshape(-1, 5)                # (0, 5) se o arquivo estiver vazio


def caixas_px(rot, largura, altura):
    """Converte caixas YOLO normalizadas em pixels (x1, y1, x2, y2)."""
    xc, yc, w, h = rot[:, 1] * largura, rot[:, 2] * altura, rot[:, 3] * largura, rot[:, 4] * altura  # desnormaliza
    return np.stack([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2], axis=1)  # centro/tamanho → cantos


linhas = []                                                            # uma linha por (split, classe)
for s in SPLITS:                                                       # percorre train, val e test
    por_img, inst = Counter(), Counter()                               # imagens que contêm a classe / nº de caixas
    for img in imagens(s):                                             # cada imagem do split
        cls = ler_rotulos(rotulo_de(img))[:, 0].astype(int)            # classes das caixas da imagem
        inst.update(cls)                                               # cada caixa conta
        por_img.update(set(cls))                                       # cada imagem conta 1 vez por classe
    for i, c in enumerate(CLASSES):                                    # monta a linha de cada classe
        linhas.append({"split": s, "classe": c, "imagens": por_img[i], "caixas": inst[i],
                       "atende ao mínimo": por_img[i] >= MINIMO[s]})   # compara com 32/4/4

contagem = pd.DataFrame(linhas)                                        # tabela longa
display(contagem.pivot(index="classe", columns="split", values=["imagens", "caixas"])
        .reindex(CLASSES)[[(m, s) for m in ("imagens", "caixas") for s in SPLITS]])  # tabela classe × split
print({s: len(imagens(s)) for s in SPLITS}, "imagens por split")       # total de arquivos por split
print("Todas as classes atendem ao mínimo:", contagem["atende ao mínimo"].all())  # resumo do critério

nomes = {s: {p.stem for p in imagens(s)} for s in SPLITS}              # nomes de arquivo por split
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:    # cada par de splits
    print(f"Vazamento {a}×{b}: {len(nomes[a] & nomes[b])} imagens em comum")  # o esperado é 0

In [ ]:
def desenhar(img, rot):
    """Desenha as caixas rotuladas sobre a imagem (para conferência visual)."""
    im = Image.open(img).convert("RGB")                                # abre em RGB
    d = ImageDraw.Draw(im)                                             # contexto de desenho
    for (x1, y1, x2, y2), c in zip(caixas_px(rot, *im.size), rot[:, 0].astype(int)):  # cada caixa
        d.rectangle([x1, y1, x2, y2], outline=CORES[c], width=max(2, im.width // 200))  # contorno na cor da classe
        d.text((x1 + 4, y1 + 4), CLASSES[c], fill=CORES[c])            # nome da classe no canto
    return im                                                          # imagem anotada


amostra = random.Random(SEED).sample(imagens("train"), k=min(8, len(imagens("train"))))  # 8 imagens fixas pela seed
fig, axs = plt.subplots(2, 4, figsize=(16, 8))                         # grade 2 × 4
for ax in axs.flat:                                                    # desliga os eixos de todos os quadros
    ax.axis("off")                                                     # inclusive os que ficarem vazios
for ax, img in zip(axs.flat, amostra):                                 # preenche com as amostras
    ax.imshow(desenhar(img, ler_rotulos(rotulo_de(img))))              # imagem + caixas
    ax.set_title(img.name, fontsize=8)                                 # nome do arquivo
plt.suptitle("Amostras de treino com os rótulos revisados")            # título da figura
plt.tight_layout(); plt.show()                                         # exibe

> ⏳ Pendente de execução: após rodar as células acima, confiro na tabela se todas as classes atingem 32/4/4 e se não há vazamento entre os conjuntos. Também comento o desbalanceamento de **caixas** (muitos mourões por foto contra um escorpião por foto), que deve aparecer de novo nas métricas por classe.

<a id="4"></a>
## 4. Entrega 1 · Treino da YOLOv5 customizada

| Escolha | Valor | Por quê |
|---|---|---|
| Modelo | `yolov5s.pt` pré-treinado no COCO | *Transfer learning*: com cerca de 32 imagens por classe, treinar do zero não funciona. É o modelo do capítulo, é leve (cerca de 7 M de parâmetros) e é o que o Hailo Model Zoo suporta melhor (ver o Ir Além). O COCO já tem `cow` e `cup`, então `vaca` e `caneca` começam com vantagem. |
| Resolução | 640 px | padrão do YOLOv5; objetos pequenos (escorpião) precisam de pixels |
| Lote | 16 | cabe com folga na GPU do Colab (T4 ou superior) |
| `--cache` | imagens em RAM | o dataset é pequeno, e isso acelera as épocas |
| Simulações | **30 e 60 épocas** | dois valores bem diferentes, como pede o enunciado |
| Extra (opcional) | 60 épocas com `--freeze 10` | congela o *backbone* e treina só o pescoço e a cabeça, o que pode reduzir o *overfitting* |

**Onde ficam os resultados:** o enunciado cita `yolov5/runs/detect/expX`. No YOLOv5, as métricas de treino ficam em `runs/train/<nome>/` (`results.csv`, `results.png`, `weights/best.pt`) e as imagens anotadas em `runs/detect/<nome>/`. Usei `--project /content/runs` e nomes explícitos (`exp_30ep`, `exp_60ep`) no lugar de `expX`, para que cada pasta identifique o experimento.

In [ ]:
TEMPOS_JSON = RUNS / "tempos_treino.json"                              # tempos sobrevivem a uma reexecução parcial
TEMPOS = json.loads(TEMPOS_JSON.read_text()) if TEMPOS_JSON.exists() else {}  # segundos por experimento
PESOS = {}                                                             # nome do experimento → best.pt


def treinar(nome, epocas, extra=()):
    """Treina a YOLOv5s a partir do COCO e registra o tempo; pula se o run já existir."""
    best = RUNS / "train" / nome / "weights" / "best.pt"               # melhor peso (pelo fitness na validação)
    if best.exists() and nome in TEMPOS:                               # evita retreinar ao reexecutar
        print(f"{nome}: já treinado em {TEMPOS[nome] / 60:.1f} min")   # informa e reaproveita
        return best                                                    # caminho do peso
    _, seg = rodar([sys.executable, "train.py", "--img", IMG, "--batch", BATCH, "--epochs", epocas,
                    "--data", DATA_YAML, "--weights", "yolov5s.pt", "--project", RUNS / "train",
                    "--name", nome, "--exist-ok", "--cache", "--seed", SEED, *extra])  # treino
    TEMPOS[nome] = seg                                                 # guarda a duração
    TEMPOS_JSON.write_text(json.dumps(TEMPOS, indent=2))               # persiste em disco
    salvar_drive(RUNS)                                                 # backup opcional no Drive
    return best                                                        # caminho do peso

In [ ]:
PESOS["exp_30ep"] = treinar("exp_30ep", EPOCAS[0])                            # simulação 1: 30 épocas (3 no modo ensaio)

In [ ]:
PESOS["exp_60ep"] = treinar("exp_60ep", EPOCAS[1])                            # simulação 2: 60 épocas (6 no modo ensaio)

In [ ]:
if RODAR_FREEZE:                                                       # experimento extra, desligado por padrão
    PESOS["exp_60ep_freeze10"] = treinar("exp_60ep_freeze10", EPOCAS[1], ("--freeze", "10"))  # 10 camadas congeladas
print({n: f"{TEMPOS[n] / 60:.1f} min" for n in PESOS})                 # resumo dos tempos de treino

<a id="5"></a>
## 5. Entrega 1 · Validação e teste

O enunciado pede a comparação de **acurácia, erro e desempenho**. Em detecção, interpretei esses termos assim:

- **Acurácia → Precision (P), Recall (R), mAP@0.5 e mAP@0.5:0.95.** P mede quantas detecções estão certas, R mede quantos objetos foram encontrados, e o mAP resume a curva P×R. O mAP@0.5 aceita a caixa com IoU ≥ 0,5; o mAP@0.5:0.95 é a média para IoU de 0,5 a 0,95 e é mais exigente com a precisão da caixa.
- **Erro → perdas (*losses*)** de caixa, objetividade e classe, no treino e na validação.
- **Desempenho → tempo de treino e tempo de inferência por imagem.**

Rodo o `val.py` em **validação** e em **teste** para cada experimento. As tabelas são montadas a partir da saída do próprio YOLOv5, sem números digitados.

In [ ]:
COLS = ["classe", "imagens", "instâncias", "P", "R", "mAP50", "mAP50-95"]  # colunas impressas pelo val.py
PADRAO = re.compile(r"^\s*(all|" + "|".join(CLASSES) + r")\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s*$",
                    re.M)                                              # uma linha da tabela por classe + "all"
VELOC = re.compile(r"Speed: ([\d.]+)ms pre-process, ([\d.]+)ms inference, ([\d.]+)ms NMS")  # linha de tempos


def extrair(saida):
    """Transforma o log do val.py em (DataFrame por classe, dict de tempos em ms/img)."""
    saida = saida.replace("\r", "\n")                                  # neutraliza as barras do tqdm
    df = pd.DataFrame(PADRAO.findall(saida), columns=COLS)             # linhas da tabela de métricas
    df[COLS[1:]] = df[COLS[1:]].apply(pd.to_numeric)                   # texto → número
    pre, inf, nms = map(float, VELOC.findall(saida)[-1])               # última linha "Speed:"
    return df, {"pre": pre, "inferência": inf, "nms": nms}             # métricas e tempos

In [ ]:
def validar(nome, task):
    """Roda o val.py do experimento num conjunto (val ou test) e devolve métricas e tempos."""
    saida, _ = rodar([sys.executable, "val.py", "--weights", PESOS[nome], "--data", DATA_YAML,
                      "--img", IMG, "--task", task, "--verbose", "--project", RUNS / "val",
                      "--name", f"{nome}_{task}", "--exist-ok"])       # salva gráficos em runs/val/<nome>_<task>
    return extrair(saida)                                              # parse do log


glob, porcls = [], []                                                  # tabelas global e por classe
for nome in PESOS:                                                     # cada experimento treinado
    for task in ("val", "test"):                                       # validação e teste
        df, vel = validar(nome, task)                                  # roda e extrai
        g = df[df.classe == "all"].iloc[0]                             # linha agregada
        glob.append({"conjunto": task, "experimento": nome, "P": g.P, "R": g.R, "mAP50": g.mAP50,
                     "mAP50-95": g["mAP50-95"], "treino (min)": TEMPOS[nome] / 60,
                     "inferência (ms/img)": vel["inferência"], "total c/ pré e NMS (ms/img)": sum(vel.values())})
        porcls += [{**r, "conjunto": task, "experimento": nome} for r in df[df.classe != "all"].to_dict("records")]

tab_global = pd.DataFrame(glob).set_index(["conjunto", "experimento"]).round(3)  # índice duplo
tab_classe = pd.DataFrame(porcls)                                      # formato longo
tab_global.to_csv(RUNS / "metricas_global.csv"); tab_classe.to_csv(RUNS / "metricas_por_classe.csv", index=False)  # persiste
display(tab_global)                                                    # mostra tudo

In [ ]:
t = tab_global.loc["test"]                                             # só o conjunto de teste
delta = (t.loc["exp_60ep"] - t.loc["exp_30ep"]).rename("Δ (60 − 30)")  # diferença 60 − 30 épocas
display(pd.concat([t.T, delta], axis=1).round(3))                      # tabela 30 × 60 com Δ

por_classe = (tab_classe[tab_classe.conjunto == "test"]                # métricas por classe no teste
              .pivot(index="classe", columns="experimento", values=["P", "R", "mAP50", "mAP50-95"])
              .reindex(CLASSES).round(3))                              # ordem fixa das classes
display(por_classe)                                                    # tabela por classe

In [ ]:
def ler_results(nome):
    """Lê o results.csv de um treino (o YOLOv5 preenche os nomes das colunas com espaços)."""
    df = pd.read_csv(RUNS / "train" / nome / "results.csv")            # uma linha por época
    df.columns = df.columns.str.strip()                                # remove os espaços dos nomes
    return df                                                          # DataFrame limpo


PAINEIS = [("box_loss", "Perda de caixa"), ("cls_loss", "Perda de classe"),  # erro: treino × validação
           ("metrics/mAP_0.5", "mAP@0.5 (validação)"), ("metrics/mAP_0.5:0.95", "mAP@0.5:0.95 (validação)")]
fig, axs = plt.subplots(2, 2, figsize=(14, 9))                         # 4 painéis
for ax, (col, titulo) in zip(axs.flat, PAINEIS):                       # um painel por métrica
    for nome in PESOS:                                                 # sobrepõe os experimentos
        r = ler_results(nome)                                          # curvas do experimento
        if col.startswith("metrics"):                                  # métrica de validação
            ax.plot(r.epoch + 1, r[col], label=nome)                   # uma linha
        else:                                                          # perdas: treino (cheia) e validação (tracejada)
            linha, = ax.plot(r.epoch + 1, r[f"train/{col}"], label=f"{nome} treino")  # treino
            ax.plot(r.epoch + 1, r[f"val/{col}"], "--", color=linha.get_color(), label=f"{nome} val")  # validação
    ax.set_title(titulo); ax.set_xlabel("época"); ax.grid(alpha=0.3); ax.legend(fontsize=8)  # acabamento
plt.suptitle("30 × 60 épocas: a distância entre treino e validação indica overfitting")  # título geral
plt.tight_layout(); plt.savefig(ASSETS / "curvas_30x60.png", dpi=120); plt.show()  # salva e exibe

In [ ]:
for nome in PESOS:                                                     # gráficos gerados pelo YOLOv5
    figs = [RUNS / "train" / nome / "results.png", RUNS / "val" / f"{nome}_test" / "confusion_matrix.png"]  # curvas e matriz
    fig, axs = plt.subplots(1, 2, figsize=(18, 6))                     # lado a lado
    for ax, f in zip(axs, figs):                                       # cada figura
        ax.imshow(Image.open(f)); ax.axis("off"); ax.set_title(f"{nome} · {f.parent.name}/{f.name}")  # exibe
    plt.tight_layout(); plt.show()                                     # mostra o par

> ⏳ Pendente de execução: após o treino, analiso os seguintes pontos.
> - **30 × 60 (global):** se o mAP@0.5 do teste subiu, ficou igual ou caiu com 60 épocas, e quanto o treino ficou mais caro em minutos.
> - **Erro:** se, nas curvas, a perda de validação parou de cair (ou subiu) enquanto a de treino continuou caindo, e em que época isso ocorreu. Esse comportamento indica *overfitting*. Com cerca de 32 imagens por classe ele é esperado, e o `best.pt` (escolhido pela validação) atenua o efeito.
> - **Por classe:** comparo `vaca`, `arvore` e `caneca` (objetos que ocupam boa parte da foto; `vaca` e `caneca` já existem no COCO) com `escorpiao` (objeto pequeno, poucos pixels na caixa) e `cerca` (para verificar se a estratégia do mourão funcionou).
> - **Matriz de confusão:** quais classes se confundem entre si e quais se confundem com o fundo (*background*).

<a id="6"></a>
## 6. Entrega 1 · Inferência nas imagens de teste

Escolho o melhor experimento pelo **mAP@0.5 da validação**, não do teste. Escolher pelo teste seria usar o próprio teste para tomar uma decisão de modelo, o que deixaria a métrica final otimista. O `detect.py` processa as imagens de teste com confiança mínima de 0,25.

In [ ]:
MELHOR = max(PESOS, key=lambda n: tab_global.loc[("val", n), "mAP50"])  # escolha pela validação
print("Melhor experimento (validação):", MELHOR)                       # mostra a escolha
rodar([sys.executable, "detect.py", "--weights", PESOS[MELHOR], "--source", DATA_DIR / "test" / "images",
       "--img", IMG, "--conf-thres", 0.25, "--project", RUNS / "detect", "--name", "teste_custom",
       "--exist-ok", "--line-thickness", 2])                           # salva em runs/detect/teste_custom

saidas = sorted((RUNS / "detect" / "teste_custom").glob("*.jpg"))      # imagens anotadas pelo modelo
n = min(12, len(saidas)); colunas = 4; linhas_g = -(-n // colunas)     # até 12 imagens em grade de 4 colunas
fig, axs = plt.subplots(linhas_g, colunas, figsize=(18, 4.5 * linhas_g), squeeze=False)  # grade
for ax in axs.flat:                                                    # desliga todos os eixos
    ax.axis("off")                                                     # inclusive os vazios
for ax, f in zip(axs.flat, saidas[:n]):                                # preenche a grade
    ax.imshow(Image.open(f)); ax.set_title(f.name, fontsize=8)         # imagem processada
    shutil.copy2(f, ASSETS / f"teste_{f.name}")                        # cópia para o README e o vídeo
plt.suptitle(f"Imagens de teste processadas pelo modelo customizado ({MELHOR})")  # título
plt.tight_layout(); plt.savefig(ASSETS / "teste_custom_grade.jpg", dpi=110); plt.show()  # salva e exibe
salvar_drive(RUNS, ASSETS)                                             # backup opcional

<a id="7"></a>
## 7. Entrega 1 · Conclusões

> ⏳ Pendente de execução: com os números das tabelas acima, escrevo as conclusões nestes quatro pontos.
> 1. **Se mais épocas compensaram:** comparo o mAP@0.5 e o mAP@0.5:0.95 do teste com o custo de treino, e concluo pela evidência, sem partir da ideia de que mais épocas é sempre melhor.
> 2. **Onde o modelo é confiável:** as classes com mAP alto e o que elas têm em comum (tamanho do objeto, contraste com o fundo, presença no COCO).
> 3. **Onde não é:** as classes fracas e a causa provável (tamanho em pixels, camuflagem no capim, poucas instâncias).
> 4. **Para o cliente:** uma imagem de teste em que vaca, árvore e cerca aparecem juntas e o que o rover faria com essa leitura.

<a id="8"></a>
## 8. Entrega 2a · YOLO tradicional (COCO, sem customização)

**Interpretação:** considerei como "YOLO tradicional" o `yolov5s` **pré-treinado no COCO e usado como vem**, sem treino com o meu dataset. É o mesmo ponto de partida da Entrega 1, então a comparação isola o efeito da customização.

> ⏳ Pendente: vou confirmar no capítulo 3 de Redes Neurais se o "YOLO padrão" do curso corresponde a esse modelo.

O COCO tem 80 classes. Das seis que defini, duas existem lá: `vaca` (`cow`) e `caneca` (`cup`, que no COCO abrange xícaras e canecas). As outras quatro, justamente as do risco biológico e dos limites do pasto, não existem. Para medir a precisão com **a mesma régua nos dois YOLOs**, implementei uma função simples de AP@0.5 e a apliquei aos rótulos de teste:

- **COCO:** `cow` é mapeada para `vaca` e `cup` para `caneca`, e calculo o AP@0.5 de cada uma. As outras quatro classes têm AP = 0 por construção, porque o modelo não tem saída para elas. Além da média nas seis classes, a tabela traz a média só em `vaca` e `caneca`, que mostra o desempenho do COCO nas classes que ele conhece.
- **Customizada:** a mesma função, classe a classe. Ela também serve para conferir o resultado do `val.py`.

In [ ]:
def iou(caixa, outras):
    """IoU entre uma caixa (4,) e várias (n, 4), formato x1, y1, x2, y2."""
    x1 = np.maximum(caixa[0], outras[:, 0]); y1 = np.maximum(caixa[1], outras[:, 1])  # canto sup. esq. da interseção
    x2 = np.minimum(caixa[2], outras[:, 2]); y2 = np.minimum(caixa[3], outras[:, 3])  # canto inf. dir. da interseção
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)      # área da interseção (0 se não sobrepõe)
    area = lambda b: (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])  # área de caixas
    return inter / (area(caixa) + area(outras) - inter + 1e-9)          # interseção / união


def ap50(preds, gts):
    """AP@0.5 de uma classe. preds: por imagem, (k, 5) [x1,y1,x2,y2,conf]; gts: por imagem, (m, 4)."""
    n_obj = sum(len(g) for g in gts)                                   # total de objetos reais
    if n_obj == 0:                                                     # classe ausente no conjunto
        return float("nan")                                            # AP indefinido
    regs = sorted(((p[4], i, p[:4]) for i, ps in enumerate(preds) for p in ps), key=lambda r: -r[0])  # por confiança
    usado = [np.zeros(len(g), bool) for g in gts]                      # cada objeto real casa uma única vez
    tp = np.zeros(len(regs))                                           # 1 = verdadeiro positivo
    for k, (_, i, caixa) in enumerate(regs):                           # da detecção mais confiante para a menos
        if len(gts[i]):                                                # há objetos reais nesta imagem
            ious = iou(caixa, gts[i]); j = ious.argmax()               # objeto real mais sobreposto
            if ious[j] >= 0.5 and not usado[i][j]:                     # casou e ainda estava livre
                usado[i][j] = True; tp[k] = 1                          # conta como acerto
    tpc, fpc = np.cumsum(tp), np.cumsum(1 - tp)                        # acumulados ao longo do ranking
    rec = tpc / n_obj; prec = tpc / np.maximum(tpc + fpc, 1e-9)        # curva precisão × revocação
    mrec = np.concatenate([[0.0], rec, [1.0]]); mpre = np.concatenate([[1.0], prec, [0.0]])  # sentinelas
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))               # envelope monotônico da precisão
    idx = np.where(mrec[1:] != mrec[:-1])[0]                           # pontos onde a revocação muda
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))  # área sob a curva (todos os pontos)

In [ ]:
TESTE = imagens("test")                                                # imagens de teste
ARR = [np.array(Image.open(p).convert("RGB")) for p in TESTE]          # carregadas uma vez (RGB, HWC)
GT = []                                                                # rótulos de teste em pixels, por imagem
for p, a in zip(TESTE, ARR):                                           # cada imagem
    r = ler_rotulos(rotulo_de(p))                                      # caixas normalizadas
    GT.append((r[:, 0].astype(int), caixas_px(r, a.shape[1], a.shape[0])))  # (classes, caixas em px)

carregar = lambda pesos: torch.hub.load(str(YOLO_DIR), "custom", path=str(pesos), source="local", verbose=False)  # AutoShape
coco = carregar(YOLO_DIR / "yolov5s.pt")                               # YOLO tradicional (baixado no treino)
custom = carregar(PESOS[MELHOR])                                       # YOLO customizada da Entrega 1


def prever(modelo, conf):
    """Detecções por imagem de teste: array (k, 6) [x1, y1, x2, y2, conf, classe] em pixels."""
    modelo.conf = conf                                                 # limiar de confiança do AutoShape
    return [modelo(a, size=IMG).xyxy[0].cpu().numpy() for a in ARR]    # NMS já aplicado


def ap_classe(dets, cls_modelo, cls_gt):
    """AP@0.5 de uma classe do dataset a partir das detecções de uma classe do modelo."""
    preds = [d[d[:, 5] == cls_modelo][:, :5] for d in dets]            # só a classe do modelo
    gts = [cx[c == cls_gt] for c, cx in GT]                            # só a classe do dataset
    return ap50(preds, gts)                                            # AP@0.5


DET_COCO, DET_CUSTOM = prever(coco, 0.001), prever(custom, 0.001)      # limiar baixo = curva P×R completa
COCO_MAP = {"vaca": "cow", "caneca": "cup"}                            # classes do dataset que existem no COCO
COCO_IDX = {c: next(k for k, v in coco.names.items() if v == n) for c, n in COCO_MAP.items()}  # índice de cada uma no COCO
ap = pd.DataFrame({"YOLOv5s COCO": [ap_classe(DET_COCO, COCO_IDX[c], i) if c in COCO_IDX else 0.0
                                    for i, c in enumerate(CLASSES)],   # AP = 0 nas classes sem correspondente
                   "YOLOv5s customizada": [ap_classe(DET_CUSTOM, i, i) for i in range(len(CLASSES))]},
                  index=CLASSES)                                       # mesma função para os dois
ap.loc[[c for i, c in enumerate(CLASSES) if not any((g[0] == i).any() for g in GT)]] = np.nan  # classe sem objeto no teste
ap["val.py (custom, teste)"] = por_classe[("mAP50", MELHOR)]           # conferência com o YOLOv5
ap_cls = ap.loc[CLASSES]                                               # só as linhas de classe
ap.loc["mAP@0.5 (média)"] = ap_cls.mean()                              # média nas classes presentes
ap.loc["mAP@0.5 (vaca e caneca)"] = ap_cls.loc[list(COCO_MAP)].mean()  # só as classes que o COCO conhece
display(ap.round(3))                                                   # tabela final

In [ ]:
COCO_PARA_NOSSA = []                                                   # o que o COCO "vê" em cada objeto rotulado
for (cls, caixas), d in zip(GT, prever(coco, 0.25)):                   # detecções com confiança de uso real
    for c, cx in zip(cls, caixas):                                     # cada objeto rotulado
        ious = iou(cx, d[:, :4]) if len(d) else np.zeros(0)            # sobreposição com as detecções
        rotulo = coco.names[int(d[ious.argmax(), 5])] if len(ious) and ious.max() >= 0.3 else "(nada)"  # melhor casamento
        COCO_PARA_NOSSA.append({"classe real": CLASSES[c], "COCO detectou": rotulo})  # registra o par

tab = pd.crosstab(pd.DataFrame(COCO_PARA_NOSSA)["classe real"], pd.DataFrame(COCO_PARA_NOSSA)["COCO detectou"])
display(tab.reindex([c for c in CLASSES if c in tab.index]))           # classe real × rótulo do COCO

render = coco([a for a in ARR[:8]], size=IMG).render()                 # desenha as detecções do COCO (conf 0,25)
fig, axs = plt.subplots(2, 4, figsize=(18, 8))                         # grade 2 × 4
for ax in axs.flat:                                                    # desliga todos os eixos
    ax.axis("off")                                                     # inclusive os vazios
for ax, im, p in zip(axs.flat, render, TESTE):                         # preenche
    ax.imshow(im); ax.set_title(p.name, fontsize=8)                    # imagem anotada pelo COCO
plt.suptitle("YOLOv5s COCO, sem customização, nas mesmas imagens de teste")  # título
plt.tight_layout(); plt.savefig(ASSETS / "teste_coco_grade.jpg", dpi=110); plt.show()  # salva e exibe

> ⏳ Pendente de execução: verifico se o COCO acerta `vaca` e `caneca` sem nenhum treino e comparo esses APs com os do modelo customizado, que aprendeu as mesmas classes com poucas dezenas de imagens. Na tabela cruzada, observo o que o COCO "vê" nas cobras, nos escorpiões e nos mourões (nada, `bird`, `bench`?). Esse é o argumento central da customização: **o modelo COCO não tem classe para os riscos que o rover precisa detectar**, e a média restrita a `vaca` e `caneca` mostra o que ele entrega quando a classe existe.

<a id="9"></a>
## 9. Entrega 2b · CNN treinada do zero

**O problema:** classificação atribui **uma classe por imagem**, mas as cenas de pasto têm vários objetos (vaca + árvore + mourão).
**A solução que adotei:** gerar o dataset de classificação **recortando as caixas já rotuladas**. Cada recorte vira uma amostra de classe única. Assim aproveito toda a rotulação e aumento o número de amostras.

- Recorte **quadrado** em volta da caixa, com 10% de margem, sem distorcer a proporção (um mourão continua fino e alto), redimensionado para 128×128.
- Cada recorte **herda o split da imagem de origem**, então não há vazamento.
- **`class_weight`** compensa o desbalanceamento (há muitos mourões por foto e um escorpião por foto).
- *Augmentation* (espelhamento, rotação, zoom e contraste) só no treino, porque essas camadas ficam inativas na inferência.
- Arquitetura: 3 blocos **Conv → BatchNorm → ReLU → MaxPool** (32, 64 e 128 filtros) → *Global Average Pooling* → *Dropout* → *softmax* de 6 classes. Mantive a rede pequena de propósito, porque são poucas centenas de amostras.

> **Ressalva:** classificar um recorte **não é** detectar. A CNN recebe o objeto já localizado por um rótulo feito à mão. Para o rover isso não basta, porque ele precisa saber **onde** está a cobra. Por isso a comparação da seção 12 é feita com essa ressalva.

In [ ]:
import tensorflow as tf                                                # Keras para a CNN
from tensorflow.keras import layers                                    # camadas
for gpu in tf.config.list_physical_devices("GPU"):                     # divide a GPU com o PyTorch
    tf.config.experimental.set_memory_growth(gpu, True)                # aloca memória sob demanda

LADO, MARGEM = 128, 0.10                                               # tamanho do recorte e margem em volta da caixa


def recortes(split):
    """Recorta cada caixa rotulada do split → (X uint8 [n,128,128,3], y int [n])."""
    X, y = [], []                                                      # amostras e rótulos
    for img in imagens(split):                                         # cada imagem do split
        im = Image.open(img).convert("RGB")                            # abre em RGB
        rot = ler_rotulos(rotulo_de(img))                              # caixas da imagem
        for c, (x1, y1, x2, y2) in zip(rot[:, 0].astype(int), caixas_px(rot, *im.size)):  # cada caixa
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2                      # centro da caixa
            meio = max(x2 - x1, y2 - y1) * (1 + 2 * MARGEM) / 2        # meio lado do quadrado com margem
            quadro = (cx - meio, cy - meio, cx + meio, cy + meio)      # quadrado (pode sair da imagem → preto)
            X.append(np.asarray(im.crop(quadro).resize((LADO, LADO))))  # recorta e redimensiona
            y.append(c)                                                # classe do recorte
    return np.stack(X), np.array(y)                                    # arrays prontos para o Keras

In [ ]:
(Xtr, ytr), (Xva, yva), (Xte, yte) = (recortes(s) for s in SPLITS)     # recortes herdam o split da imagem
display(pd.DataFrame({s: np.bincount(y, minlength=len(CLASSES)) for s, y in zip(SPLITS, (ytr, yva, yte))},
                     index=CLASSES))                                   # nº de recortes por classe e split

from sklearn.utils.class_weight import compute_class_weight            # pesos inversos à frequência
presentes = np.unique(ytr)                                             # classes com amostras no treino
PESOS_CLASSE = dict(zip(presentes.tolist(), compute_class_weight("balanced", classes=presentes, y=ytr)))  # {classe: peso}
print("class_weight:", {CLASSES[k]: round(float(v), 2) for k, v in PESOS_CLASSE.items()})  # mostra os pesos

fig, axs = plt.subplots(1, len(CLASSES), figsize=(15, 3))              # um exemplo por classe
for ax, i in zip(axs, range(len(CLASSES))):                            # cada classe
    ax.axis("off"); ax.set_title(CLASSES[i])                           # título
    if (ytr == i).any():                                               # se houver recorte desta classe
        ax.imshow(Xtr[ytr == i][0])                                    # primeiro recorte
plt.show()                                                             # exibe

In [ ]:
tf.keras.utils.set_random_seed(SEED)                                   # pesos iniciais reprodutíveis
aumento = tf.keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.1),
                               layers.RandomZoom(0.15), layers.RandomContrast(0.2)], name="augmentation")  # só no treino


def bloco(x, filtros):
    """Conv 3×3 → BatchNorm → ReLU → MaxPool (reduz a resolução pela metade)."""
    x = layers.Conv2D(filtros, 3, padding="same", use_bias=False)(x)   # extrai padrões locais
    x = layers.BatchNormalization()(x)                                 # estabiliza o treino
    x = layers.ReLU()(x)                                               # não linearidade
    return layers.MaxPooling2D()(x)                                    # 2× menor


entrada = layers.Input((LADO, LADO, 3))                                # imagem RGB 128×128
x = layers.Rescaling(1 / 255)(aumento(entrada))                        # augmentation + escala para [0, 1]
for filtros in (32, 64, 128):                                          # três blocos convolucionais
    x = bloco(x, filtros)                                              # 128 → 64 → 32 → 16 px
x = layers.GlobalAveragePooling2D()(x)                                 # um valor por mapa (poucos parâmetros)
x = layers.Dropout(0.3)(x)                                             # regularização
saida = layers.Dense(len(CLASSES), activation="softmax")(x)            # probabilidade de cada classe
cnn = tf.keras.Model(entrada, saida, name="cnn_do_zero")               # modelo completo
cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])                                      # rótulos inteiros
cnn.summary()                                                          # arquitetura e nº de parâmetros

In [ ]:
t0 = time.perf_counter()                                               # cronômetro do treino
hist = cnn.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCAS_CNN, batch_size=32,
               class_weight=PESOS_CLASSE, verbose=2,
               callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                                           restore_best_weights=True)])  # para quando a validação estagna
TEMPOS["cnn"] = time.perf_counter() - t0                               # duração do treino
TEMPOS_JSON.write_text(json.dumps(TEMPOS, indent=2))                   # persiste junto com os do YOLO

fig, axs = plt.subplots(1, 2, figsize=(13, 4))                         # perda e acurácia
for ax, m in zip(axs, ("loss", "accuracy")):                           # cada métrica
    ax.plot(hist.history[m], label="treino"); ax.plot(hist.history[f"val_{m}"], "--", label="validação")  # curvas
    ax.set_title(m); ax.set_xlabel("época"); ax.grid(alpha=0.3); ax.legend()  # acabamento
plt.suptitle(f"CNN do zero · {len(hist.history['loss'])} épocas · {TEMPOS['cnn'] / 60:.1f} min"); plt.show()  # exibe

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report  # avaliação de classificação

pred = cnn.predict(Xte, verbose=0).argmax(1)                           # classe mais provável de cada recorte
ACC_CNN = float((pred == yte).mean())                                  # acurácia no teste
print(f"Acurácia no teste: {ACC_CNN:.3f} ({(pred == yte).sum()}/{len(yte)} recortes)")  # resumo
print(classification_report(yte, pred, labels=range(len(CLASSES)), target_names=CLASSES, zero_division=0))  # P/R/F1
ConfusionMatrixDisplay.from_predictions(yte, pred, labels=range(len(CLASSES)), display_labels=CLASSES,
                                        cmap="Greens", colorbar=False)  # matriz de confusão
plt.title("CNN do zero · matriz de confusão (teste)"); plt.savefig(ASSETS / "cnn_confusao.png", dpi=110); plt.show()  # salva e exibe

> ⏳ Pendente de execução: após o treino, verifico se a CNN separa bem as classes grandes (`vaca`, `arvore`) e quais classes se confundem (por exemplo, cobra × mourão, que são formas alongadas). Também observo se, com poucas centenas de recortes, a curva de validação oscila e se o *early stopping* interrompeu o treino cedo.

<a id="10"></a>
## 10. Entrega 2 · Extra: YOLOv8s no mesmo dataset

Incluí na comparação uma linha que o enunciado não pede: a **YOLOv8s** do pacote `ultralytics`, treinada no mesmo dataset, com o mesmo `data.yaml`, a mesma resolução (640 px), a mesma semente e o mesmo número de épocas da simulação 2 da Entrega 1 (60 épocas; 6 no modo ensaio). Também parte de pesos pré-treinados no COCO (`yolov8s.pt`), como a YOLOv5s.

O objetivo é comparar a geração de YOLO usada no curso com uma geração mais recente. A diferença principal de arquitetura é que a YOLOv8 é *anchor-free*: a cabeça prevê diretamente o centro do objeto e as distâncias até as bordas da caixa, sem depender das âncoras pré-definidas da YOLOv5, e separa os ramos de classificação e de regressão (*decoupled head*). Como o dataset mistura proporções muito diferentes (mourão fino e alto, vaca larga, escorpião pequeno), quis verificar se essa mudança faz diferença com tão poucas imagens.

Esta seção é complementar. O modelo usado no Ir Além (câmera Wi-Fi, VS Code e Raspberry Pi 5 com Hailo-8L) continua sendo a **YOLOv5s customizada** da Entrega 1. A flag `RODAR_YOLOV8` (seção 2) desliga a seção; nesse caso, as seções 11 e 12 comparam apenas os três modelos do enunciado.

- **Treino:** `model.train()` com o mesmo `DATA_YAML` da seção 3, salvo em `runs/train/yolov8s_60ep`. O tempo é cronometrado em volta da chamada, como no YOLOv5.
- **Avaliação:** `model.val(split="test")`, que calcula P, R, mAP@0.5 e mAP@0.5:0.95 com as mesmas definições do `val.py` do YOLOv5. Mostro o resultado ao lado do melhor experimento da Entrega 1.
- **Inferência:** entra no protocolo comum da seção 11, com a mesma função de cronometragem e a imagem inteira.

In [ ]:
V8_NOME = "yolov8s_60ep"                                               # nome do run (EPOCAS[1]: 60 épocas, 6 no ensaio)
V8_BEST = RUNS / "train" / V8_NOME / "weights" / "best.pt"             # melhor peso pela validação, como no YOLOv5
if RODAR_YOLOV8:                                                       # comparação extra (flag na seção 2)
    from ultralytics import YOLO                                       # API da YOLOv8 (instalada no setup)
    if V8_BEST.exists() and V8_NOME in TEMPOS:                         # evita retreinar ao reexecutar
        print(f"{V8_NOME}: já treinado em {TEMPOS[V8_NOME] / 60:.1f} min")  # reaproveita o run
    else:                                                              # primeiro treino
        t0 = time.perf_counter()                                       # cronômetro do treino
        modelo_v8 = YOLO("yolov8s.pt")                                 # pré-treinado no COCO, como o yolov5s.pt
        modelo_v8.train(data=str(DATA_YAML), epochs=EPOCAS[1], imgsz=IMG, batch=BATCH,  # mesmo data.yaml, épocas e resolução
                        seed=SEED, deterministic=True, cache=True,     # mesma semente; imagens em RAM
                        device=0 if DEVICE == "cuda" else "cpu",       # GPU 0 do Colab
                        project=str(RUNS / "train"), name=V8_NOME, exist_ok=True)  # salvo ao lado dos runs do YOLOv5
        TEMPOS[V8_NOME] = time.perf_counter() - t0                     # duração do treino
        V8_BEST = Path(modelo_v8.trainer.best)                         # caminho real do best.pt
        TEMPOS_JSON.write_text(json.dumps(TEMPOS, indent=2))           # persiste junto com os demais tempos
        torch.use_deterministic_algorithms(False)                      # o treino liga o modo determinístico global do PyTorch;
        torch.backends.cudnn.deterministic = False                     # volto ao padrão para não afetar os tempos da seção 11
        salvar_drive(RUNS)                                             # backup opcional no Drive
    v8 = YOLO(str(V8_BEST))                                            # modelo treinado, usado na avaliação e nos tempos
    print(f"YOLOv8s: {TEMPOS[V8_NOME] / 60:.1f} min de treino · {V8_BEST}")  # resumo

In [ ]:
if RODAR_YOLOV8:                                                       # comparação extra (flag na seção 2)
    m8 = v8.val(data=str(DATA_YAML), split="test", imgsz=IMG, batch=BATCH,  # conjunto de teste, mesma resolução
                project=str(RUNS / "val"), name=f"{V8_NOME}_test", exist_ok=True)  # gráficos em runs/val/yolov8s_60ep_test
    MET_V8 = {                                                         # mesmas colunas de tab_global (seção 5)
        "P": m8.box.mp, "R": m8.box.mr, "mAP50": m8.box.map50, "mAP50-95": m8.box.map,  # médias nas classes
        "treino (min)": TEMPOS[V8_NOME] / 60, "inferência (ms/img)": m8.speed["inference"],  # tempos do model.val
        "total c/ pré e NMS (ms/img)": m8.speed["preprocess"] + m8.speed["inference"] + m8.speed["postprocess"]}  # pré + rede + NMS
    glob_v8 = pd.DataFrame({f"YOLOv5s ({MELHOR})": tab_global.loc[("test", MELHOR)],  # melhor YOLOv5 da Entrega 1
                            "YOLOv8s": pd.Series(MET_V8, dtype=float)}).round(3)  # lado a lado, no teste
    display(glob_v8)                                                   # tabela global

    tab_v8 = pd.DataFrame([{"classe": CLASSES[c], **dict(zip(("P", "R", "mAP50", "mAP50-95"), map(float, m8.box.class_result(k))))}
                           for k, c in enumerate(m8.box.ap_class_index)]).set_index("classe")  # só classes com objeto no teste
    tab_v8 = tab_v8.reindex(CLASSES)                                   # ordem fixa (NaN se a classe faltar)
    glob_v8.to_csv(RUNS / "metricas_yolov8s_global.csv"); tab_v8.to_csv(RUNS / "metricas_yolov8s_por_classe.csv")  # persiste
    display(pd.concat({f"YOLOv5s ({MELHOR})": por_classe.xs(MELHOR, axis=1, level=1), "YOLOv8s": tab_v8},
                      axis=1).round(3))                                # por classe: YOLOv5s × YOLOv8s no teste

> ⏳ Pendente de execução: comparo a YOLOv8s com a melhor YOLOv5s da Entrega 1 no mesmo conjunto de teste. Observo se o mAP@0.5 e o mAP@0.5:0.95 mudam de forma relevante, em quais classes (minha expectativa é que a cabeça *anchor-free* ajude mais em `escorpiao` e `cerca`, cujas caixas fogem das proporções comuns) e quanto o treino custou a mais ou a menos. Com cerca de 32 imagens por classe, diferenças de poucos centésimos de mAP ficam dentro da variação esperada e não bastam para concluir que uma geração é melhor que a outra.

<a id="11"></a>
## 11. Protocolo comum de tempo de inferência

Para comparar os tempos de forma justa, defini um protocolo único para todos os modelos: **a mesma GPU, as mesmas imagens já carregadas em memória, 10 inferências de aquecimento** (descartadas, porque a primeira chamada compila *kernels* e aloca memória) e depois a **média ± desvio de uma inferência por vez** no conjunto de teste, com sincronização da GPU antes de parar o cronômetro.

- **YOLOs:** tempo por **imagem inteira**, incluindo pré-processamento e NMS (o `AutoShape` no YOLOv5 e o `predict` no `ultralytics`, no caso da YOLOv8s), com confiança 0,25 (ponto de operação real). O `ultralytics` interpreta arrays NumPy como BGR, então converto as imagens uma vez, antes do cronômetro.
- **CNN:** tempo por **recorte**. Uma cena com *n* objetos custaria *n* inferências, mais um detector para encontrar os recortes. Envolvi a chamada da CNN em `tf.function`, que compila o grafo na primeira execução e o reutiliza nas seguintes. No modo *eager* o TensorFlow tem um custo fixo por chamada que, numa rede pequena como esta, domina a medição; sem a compilação eu estaria cronometrando o overhead do framework, e não a rede.

Esses tempos não medem o mesmo trabalho. O dos YOLOs inclui o pré-processamento (`letterbox`) e o NMS sobre a imagem inteira, enquanto o da CNN cobre só a classificação de um recorte já pronto. Por isso a comparação é da latência por chamada de cada abordagem, e não do custo equivalente para resolver a mesma tarefa.

In [ ]:
def sincronizar():
    """Espera a GPU terminar o trabalho pendente (senão o cronômetro mede só o enfileiramento)."""
    if torch.cuda.is_available():                                      # só faz sentido com GPU
        torch.cuda.synchronize()                                       # barreira do PyTorch


def cronometrar(fn, entradas, aquecimento=10):
    """Média e desvio (ms) de fn(x) para cada x, após o aquecimento."""
    for i in range(aquecimento):                                       # aquecimento descartado
        fn(entradas[i % len(entradas)])                                # repete as entradas se houver poucas
    sincronizar()                                                      # começa com a GPU ociosa
    ms = []                                                            # tempos individuais
    for x in entradas:                                                 # uma inferência por vez
        t0 = time.perf_counter(); fn(x); sincronizar()                 # mede até a GPU terminar
        ms.append((time.perf_counter() - t0) * 1000)                   # em milissegundos
    return float(np.mean(ms)), float(np.std(ms))                       # média e desvio


inferir_cnn = tf.function(lambda x: cnn(x, training=False), reduce_retracing=True)  # grafo compilado: mede a rede, não o custo fixo do modo eager
coco.conf = custom.conf = 0.25                                         # mesmo ponto de operação nos dois YOLOv5
TEMPO_INF = {}                                                         # modelo → (média, desvio) em ms
TEMPO_INF["YOLOv5s customizada"] = cronometrar(lambda a: custom(a, size=IMG), ARR)  # imagem inteira
if RODAR_YOLOV8:                                                       # YOLOv8s no mesmo protocolo
    ARR_BGR = [np.ascontiguousarray(a[..., ::-1]) for a in ARR]        # o ultralytics espera BGR (convenção do OpenCV)
    TEMPO_INF["YOLOv8s customizada"] = cronometrar(lambda a: v8(a, imgsz=IMG, conf=0.25, verbose=False), ARR_BGR)  # imagem inteira
TEMPO_INF["YOLOv5s COCO"] = cronometrar(lambda a: coco(a, size=IMG), ARR)  # imagem inteira
TEMPO_INF["CNN do zero"] = cronometrar(lambda x: inferir_cnn(x[None]).numpy(), list(Xte))  # .numpy() sincroniza o TF
display(pd.DataFrame(TEMPO_INF, index=["média (ms)", "desvio (ms)"]).T.round(2))  # tabela de tempos

<a id="12"></a>
## 12. Entrega 2 · Comparação final

Os quatro critérios do enunciado são facilidade de uso/integração, precisão, tempo de treino/customização e tempo de inferência. A tabela é montada a partir das variáveis calculadas nas seções anteriores. Só a facilidade de uso é qualitativa. Com `RODAR_YOLOV8` ligada, a tabela e os gráficos ganham uma coluna extra com a YOLOv8s da seção 10, ao lado da YOLOv5s customizada. A última coluna fica reservada para o *plus* do projeto: o mesmo modelo customizado rodando quantizado em INT8 no acelerador Hailo-8L de um Raspberry Pi 5, que está no notebook do Ir Além.

In [ ]:
map_custom = float(tab_global.loc[("test", MELHOR), "mAP50"])          # mAP@0.5 de teste pelo val.py
fmt = lambda k: f"{TEMPO_INF[k][0]:.1f} ± {TEMPO_INF[k][1]:.1f}"       # "média ± desvio"
CRIT = ["Tarefa", "Facilidade de uso/integração", "Precisão (teste)", "Tempo de treino/customização",
        "Tempo de inferência (ms)"]                                    # linhas da tabela
comparacao = pd.DataFrame({
    "YOLOv5s customizada (Entrega 1)": [
        "detecção: onde + o quê",
        "Média: exige rotular caixas e treinar; depois, uma linha para inferir (torch.hub ou detect.py)",
        f"mAP@0.5 = {map_custom:.3f} ({len(CLASSES)} classes)",
        f"{TEMPOS[MELHOR] / 60:.1f} min ({MELHOR}, GPU)",
        f"{fmt('YOLOv5s customizada')} por imagem"],
    "YOLOv5s COCO (Entrega 2a)": [
        "detecção, só nas 80 classes do COCO",
        "Alta: zero treino, zero rótulo; serve só se o objeto já existe no COCO",
        f"AP@0.5 vaca = {ap.loc['vaca', 'YOLOv5s COCO']:.3f}; caneca = {ap.loc['caneca', 'YOLOv5s COCO']:.3f}; "
        f"mAP@0.5 = {ap.loc['mAP@0.5 (média)', 'YOLOv5s COCO']:.3f} ({len(CLASSES)} classes)",
        "0 (usado como vem)",
        f"{fmt('YOLOv5s COCO')} por imagem"],
    "CNN do zero (Entrega 2b)": [
        "classificação de recorte: só o quê",
        "Baixa: arquitetura, pré-processamento e recortes à mão; ainda precisa de um detector",
        f"acurácia = {ACC_CNN:.3f} (recortes)",
        f"{TEMPOS['cnn'] / 60:.1f} min (GPU)",
        f"{fmt('CNN do zero')} por recorte"],
    "YOLOv5 custom @ Hailo-8L INT8 (plus)": ["ver notebook Ir Além"] * len(CRIT),
}, index=CRIT)                                                         # critérios nas linhas
if RODAR_YOLOV8:                                                       # coluna extra, ao lado da YOLOv5s customizada
    comparacao.insert(1, "YOLOv8s customizada (extra)", [
        "detecção: onde + o quê",
        "Média: mesmo dataset e mesmo data.yaml da YOLOv5; API Python única para treinar, validar e inferir",
        f"mAP@0.5 = {MET_V8['mAP50']:.3f} ({len(CLASSES)} classes)",
        f"{TEMPOS[V8_NOME] / 60:.1f} min ({EPOCAS[1]} épocas, GPU)",
        f"{fmt('YOLOv8s customizada')} por imagem"])
comparacao.to_csv(RUNS / "comparacao_final.csv")                       # persiste
with pd.option_context("display.max_colwidth", None):                  # não corta o texto das células
    display(comparacao)                                                # tabela final

In [ ]:
modelos = list(TEMPO_INF)                                              # ordem: YOLOv5 custom, YOLOv8 (se ligada), COCO, CNN
PREC = {"YOLOv5s customizada": (map_custom, "mAP@0.5"),                # modelo → (valor, métrica)
        "YOLOv5s COCO": (ap.loc["mAP@0.5 (média)", "YOLOv5s COCO"], "mAP@0.5"),
        "CNN do zero": (ACC_CNN, "acurácia\n(recortes)")}
if RODAR_YOLOV8:                                                       # comparação extra
    PREC["YOLOv8s customizada"] = (MET_V8["mAP50"], "mAP@0.5")         # mAP@0.5 do model.val no teste
COR = {"YOLOv5s customizada": "#2e6a42", "YOLOv8s customizada": "#c07a2c",  # uma cor fixa por modelo
       "YOLOv5s COCO": "#8aa892", "CNN do zero": "#3d55a8"}
precisao = [PREC[m][0] for m in modelos]                               # métricas de precisão
rotulos = [PREC[m][1] for m in modelos]                                # cada barra diz qual métrica é
cores = [COR[m] for m in modelos]                                      # cores na ordem das barras
fig, axs = plt.subplots(1, 2, figsize=(16, 4.5))                       # dois painéis
b = axs[0].bar(modelos, [TEMPO_INF[m][0] for m in modelos], yerr=[TEMPO_INF[m][1] for m in modelos],
               color=cores, capsize=4)                                 # tempo médio ± desvio
axs[0].bar_label(b, fmt="%.1f ms"); axs[0].set_title("Tempo de inferência (menor é melhor)")  # rótulos
b = axs[1].bar(modelos, precisao, color=cores)                         # precisão
axs[1].bar_label(b, labels=[f"{v:.2f}\n{r}" for v, r in zip(precisao, rotulos)])  # valor + métrica
axs[1].set_ylim(0, 1.15); axs[1].set_title("Precisão no teste (métricas diferentes, ver rótulos)")  # escala 0–1
for ax in axs:                                                         # acabamento comum
    ax.grid(axis="y", alpha=0.3); ax.tick_params(axis="x", labelsize=9)  # grade e fonte
plt.tight_layout(); plt.savefig(ASSETS / "comparacao.png", dpi=120); plt.show()  # salva e exibe
salvar_drive(RUNS, ASSETS)                                             # backup opcional

> ⏳ Pendente de execução: com a tabela preenchida, faço uma leitura crítica de cada critério.
> - **Facilidade:** o COCO é imediato, mas não enxerga 4 das 6 classes. A YOLO customizada custa a rotulação. A CNN exige mais trabalho manual e ainda não resolve a localização.
> - **Precisão:** comparo o mAP da YOLO customizada com o do COCO (efeito da customização), inclusive em `vaca` e `caneca`, que o COCO já conhece. A acurácia da CNN não é diretamente comparável com o mAP, porque o problema dela é mais fácil (o objeto já vem recortado).
> - **Treino:** minutos de GPU na YOLO contra os da CNN do zero.
> - **Inferência:** ms por imagem nos YOLOs contra ms por recorte na CNN. Em uma cena com *n* objetos, a CNN custaria *n* vezes esse tempo, mais o de um detector.
> - **YOLOv5s × YOLOv8s:** se a geração *anchor-free* ganha em mAP (principalmente em `escorpiao` e `cerca`) e quanto isso custa em tempo de treino e de inferência, lembrando que o modelo do Ir Além continua sendo a YOLOv5s.
> - **Veredito para o rover:** qual abordagem responde a "onde está a cobra?" em tempo real.

<a id="13"></a>
## 13. Conclusões gerais, pontos fortes e limitações

### Resultado

> ⏳ Pendente de execução: em 3 a 5 frases, registro qual modelo escolhi para o rover, com que mAP, com que latência e qual a principal ressalva por classe.

### Pontos fortes

- **Dataset desenhado para o problema real:** as classes respondem às três perguntas do rover, e a cena de pasto reúne vaca, árvore e cerca na mesma imagem, como o rover vai encontrar em campo.
- **Cerca rotulada no mourão:** caixas compactas e repetíveis em vez de caixas que são quase só fundo.
- **Avaliação com conjuntos separados:** a divisão é feita por grupos de imagens, o notebook confere que nenhum arquivo aparece em mais de um conjunto, e o melhor modelo é escolhido pela validação, não pelo teste.
- **Análise por classe**, e não só pela média global, porque as seis classes têm dificuldades muito diferentes.
- **Comparação com a mesma régua:** o AP dos dois YOLOs vem da mesma função, e os tempos vêm do mesmo protocolo.
- **Tabelas geradas por código** a partir das saídas do YOLOv5 e do Keras: reexecutar o notebook atualiza todas.

### Limitações

- **Dataset pequeno:** cerca de 32 imagens de treino por classe produzem métricas instáveis. Poucas imagens a mais ou a menos no teste mudam o mAP de forma visível.
- **`escorpiao` é um objeto pequeno:** ocupa poucos pixels dentro da caixa a 640 px. É a classe com maior risco de mAP baixo.
- **Perspectiva diferente entre os lotes:** o lote A vem do Wikimedia Commons, com fotos de pasto feitas por câmeras e enquadramentos variados, enquanto cobras e escorpiões vêm do iNaturalist, cujas fotos tendem a ser *close* do animal, tirado de perto pelo observador. Essa diferença pode levar o modelo a associar o "estilo de foto" (distância, fundo, enquadramento) à classe, em vez de aprender o objeto.
- **Risco residual de vazamento entre fotos do mesmo autor:** as fotos do Commons e do iNaturalist não trazem data e hora no nome, então o `split.py` trata cada uma como um grupo próprio. Um mesmo autor pode ter publicado mais de uma foto da mesma cena, e essas fotos podem ter ficado em conjuntos diferentes. Como a maior parte das imagens vem de autores e locais distintos, espero que o efeito seja pequeno, mas ele pode deixar as métricas de teste um pouco otimistas.
- **Rotulação semiautomática:** o Grounding DINO sugere as caixas e eu revisei todas à mão. É um desvio consciente do Make Sense indicado no enunciado (ver seção 1), que mantém o mesmo formato YOLO.
- **CNN × YOLO não é uma comparação de igual para igual:** a CNN classifica recortes já localizados, enquanto o YOLO detecta na cena inteira.
- **`caneca` fora do domínio do pasto:** a classe existe para a demonstração ao vivo e não tem relação com a operação do rover. As fotos do Commons costumam mostrar a caneca em ambiente interno, sobre mesas e bancadas, fundo que nenhuma outra classe tem. Isso facilita a separação pelo contexto e tende a deixar a métrica dessa classe otimista em relação às demais.
- **YOLOv5s × YOLOv8s não isola só a arquitetura:** treinei as duas com o mesmo dataset, a mesma resolução, a mesma semente e o mesmo número de épocas, mas cada pacote usa seus próprios hiperparâmetros padrão (otimizador, *augmentation*, função de perda). A comparação é entre as duas receitas completas, como cada uma seria usada na prática.
- **Tempos medidos na GPU do Colab:** os valores dependem da placa alocada (registrada na saída da primeira célula de código), e no rover o hardware é outro (ver o Ir Além).

### Próximos passos

- Capturar mais imagens de `cobra` e `escorpiao` com a própria câmera do rover, para reduzir a diferença de perspectiva entre os lotes.
- Treinar com resolução maior (por exemplo, 960 px) ou usar um segundo estágio para objetos pequenos.
- Rodar o modelo quantizado no Raspberry Pi 5 + Hailo-8L e medir quanto a quantização INT8 custa em mAP por classe (notebook do Ir Além).